In [9]:
from rdflib import Graph, Namespace, RDF, RDFS, URIRef
from neo4j import GraphDatabase
from pathlib import Path
from getpass import getpass

In [10]:
project_folder = Path.cwd()

if project_folder.name == "notebooks":
    project_folder = project_folder.parent

ontology_path = project_folder / "ontology" / "phase2" / "transportation_v2.owl"

g = Graph()
g.parse(str(ontology_path), format="xml")

TRAN = Namespace("http://example.org/transportation.owl#")

print("Ontology loaded successfully!")
print("Total RDF triples:", len(g))

Ontology loaded successfully!
Total RDF triples: 346


In [11]:
uri = "bolt://localhost:7687"
username = "neo4j"
password = getpass("Enter Neo4j password: ")

driver = GraphDatabase.driver(uri, auth=(username, password))

driver.verify_connectivity()

print("Connected to Neo4j successfully!")

Connected to Neo4j successfully!


In [12]:
from decimal import Decimal
classes = [
    "Road", "Junction", "Vehicle", "Signal",
    "Route", "Congestion", "Incident"
]

data_properties = {
    "Road": ["roadName", "length", "speedLimit", "lanes", "status"],
    "Junction": ["junctionId"],
    "Vehicle": ["vehicleId", "vehicleType", "hasRole"],
    "Signal": ["signalState", "signalPhase"],
    "Route": ["routeId", "routeDistance"],
    "Congestion": ["congestionLevel", "avgSpeed"],
    "Incident": ["incidentType", "severity", "startTime"]
}

# Find a class and all its subclasses
def get_subclasses(cls):
    result = {cls}

    for subclass in g.subjects(RDFS.subClassOf, cls):
        if isinstance(subclass, URIRef) and subclass not in result:
            result.update(get_subclasses(subclass))

    return result


with driver.session(database="neo4j") as session:

    for class_name in classes:

        class_uri = TRAN[class_name]
        allowed_types = get_subclasses(class_uri)

        individuals = set()

        for cls in allowed_types:
            individuals.update(g.subjects(RDF.type, cls))

        for individual in individuals:

            if not isinstance(individual, URIRef):
                continue

            name = str(individual).split("#")[-1]

            props = {"id": name}

            for prop_name in data_properties[class_name]:

                value = g.value(individual, TRAN[prop_name])

                if value is not None:

                    value = value.toPython()

                    if isinstance(value, Decimal):
                        value = float(value)

                    props[prop_name] = value

            # Match existing Phase 1 junctions by name
            if class_name == "Junction":

                props["name"] = name.replace("Junction_", "")

                session.run(
                    """
                    MERGE (n:Junction {name: $name})
                    SET n += $props
                    """,
                    name=props["name"],
                    props=props
                ).consume()

            else:

                query = f"""
                MERGE (n:{class_name} {{id: $id}})
                SET n += $props
                """

                session.run(
                    query,
                    id=name,
                    props=props
                ).consume()

            # Preserve emergency vehicle subclass labels
            if class_name == "Vehicle":

                labels = []

                if TRAN.EmergencyVehicle in allowed_types:
                    if any(
                        (individual, RDF.type, cls) in g
                        for cls in get_subclasses(TRAN.EmergencyVehicle)
                    ):
                        labels.append("EmergencyVehicle")

                if any(
                    (individual, RDF.type, cls) in g
                    for cls in get_subclasses(TRAN.Ambulance)
                ):
                    labels.append("Ambulance")

                for label in labels:
                    session.run(
                        f"""
                        MATCH (n:Vehicle {{id: $id}})
                        SET n:{label}
                        """,
                        id=name
                    ).consume()

        print(class_name, "- processed", len(individuals), "individuals")

print("\nOntology individuals transferred!")

Road - processed 8 individuals
Junction - processed 7 individuals
Vehicle - processed 5 individuals
Signal - processed 1 individuals
Route - processed 1 individuals
Congestion - processed 1 individuals
Incident - processed 1 individuals

Ontology individuals transferred!


In [13]:
relationships = {
    "ConnectedTo": "CONNECTED_TO",
    "Causes": "CAUSES",
    "LocatedAt": "LOCATED_AT",
    "Influences": "INFLUENCES",
    "Blocks": "BLOCKS",
    "Prioritizes": "PRIORITIZES"
}

with driver.session(database="neo4j") as session:

    for owl_property, neo4j_relationship in relationships.items():

        count = 0

        for subject, obj in g.subject_objects(TRAN[owl_property]):

            if not isinstance(subject, URIRef) or not isinstance(obj, URIRef):
                continue

            source = str(subject).split("#")[-1]
            target = str(obj).split("#")[-1]

            query = f"""
            MATCH (a {{id: $source}})
            MATCH (b {{id: $target}})
            MERGE (a)-[:{neo4j_relationship}]->(b)
            RETURN count(*) AS created
            """

            result = session.run(
                query,
                source=source,
                target=target
            ).single()

            count += result["created"]

        print(neo4j_relationship, "-", count, "relationships matched")

print("\nOntology relationships transferred successfully!")

CONNECTED_TO - 16 relationships matched
CAUSES - 1 relationships matched
LOCATED_AT - 2 relationships matched
INFLUENCES - 1 relationships matched
BLOCKS - 1 relationships matched
PRIORITIZES - 1 relationships matched

Ontology relationships transferred successfully!


In [14]:
driver.close()

print("Neo4j connection closed successfully!")

Neo4j connection closed successfully!
